# Set Up

In [17]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score

import joblib

# Data & Model Loading

In [18]:
X_test = pd.read_csv('../Datasets/Modeling/X_test.csv')
y_test = pd.read_csv('../Datasets/Modeling/y_test.csv')

X_test_cb = X_test.drop(['Tenure Group_Standard (1-2y)', 'Under-Utilizing Unlimited', 'Phone Service', 'Unlimited Data', 'Under 30', 'Receive Refund', 'Gender', 'Payment Method_Bank Withdrawal', 'Contract_One Year', 'Tenure Group_Loyal (2-4y)', 'Bill Shock', 'Internet Type_Cable', '30 to 64', 'Device Protection Plan', 'Tenure Group_Very Loyal (4y+)'], axis=1)
X_test_lgb = X_test.drop(['Senior Citizen', 'Under 30', 'Internet Service', 'Receive Refund', 'Tenure Group_New (0-1y)', 'Tenure Group_Very Loyal (4y+)', 'Internet Type_No Internet Service', 'Phone Service', 'Referral a Friend', 'Tenure Group_Standard (1-2y)', 'Tenure Group_Loyal (2-4y)', 'Is Alone', 'Under-Utilizing Unlimited', 'Unlimited Data', '30 to 64'], axis=1)
X_test_rf = X_test.drop(['Phone Service', 'Tenure Group_Standard (1-2y)', 'Receive Refund', 'Internet Type_Cable', 'Tenure Group_Loyal (2-4y)', 'Under 30', 'Payment Method_Mailed Check', 'Device Protection Plan', 'Under-Utilizing Unlimited', 'Is Alone', 'Unlimited Data', 'Total Refunds', 'Streaming Movies', 'Multiple Lines', 'Streaming Music'], axis=1)

cb = joblib.load('../Models/CatBoost Model.pkl')
lgb = joblib.load('../Models/LightGBM Model.pkl')
rf = joblib.load('../Models/Random Forest Model.pkl')

# Ensemble Evaluation

In [19]:
cb_proba = cb.predict_proba(X_test_cb)
lgb_proba = lgb.predict_proba(X_test_lgb)
rf_proba = rf.predict_proba(X_test_rf)

all_proba = np.array([cb_proba, lgb_proba, rf_proba])
avg_proba = np.mean(all_proba, axis=0)

y_pred = np.argmax(avg_proba, axis=1)

In [21]:
result = []

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, avg_proba[:, 1])

result.append({
    'Ensemble Model': 'CatBoost-LightGBM-Random Forest',
    'Accuracy': acc,
    'Precision': prec,
    'Recall': rec,
    'F1': f1,
    'ROC-AUC': roc_auc
})

result_df = pd.DataFrame(result).set_index('Ensemble Model')
result_df

,Accuracy,Precision,Recall,F1,ROC-AUC
Ensemble Model,,,,,
CatBoost-LightGBM-Random Forest,0.8193,0.61194,0.875445,0.720351,0.918947
